In [ ]:
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState, add_messages
from typing import TypedDict, Annotated

from app.core.config import settings

from langchain.chat_models import init_chat_model

# 定初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={"thinking":{"type":"disabled"}}
)

class MessageAgentState(TypedDict):
    messages:Annotated[list[BaseMessage],   add_messages]

def call_llm(state: MessagesState):
    ai_message = model.invoke(state["messages"])
    return {"messages":[ai_message]}

# 3.定义边，构建图
builder = StateGraph(MessagesState)

builder.add_node("llm", call_llm)

builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {
    "configurable":{
        "thread_id":"1"
    }
}

# 4.调用Graph
result_1 = graph.invoke({
    "messages":[HumanMessage(content="现在有10个苹果，有2个人，怎么平均分配?")]
},
config
)

result_2 = graph.invoke({
    "messages":[HumanMessage(content="3个人呢?")]
},
config
)
for message in result_2["messages"]:
    message.pretty_print()